# Câu 1 - Thuật toán A* Euclid

Tìm đường thoát từ phòng trung tâm ra rìa lâu đài trên ma trận `n x n`. Hai ô chỉ nối với nhau nếu có chung cạnh. Ô có giá trị `1` là đi được, ô có giá trị `0` là không đi được.

File đầu vào dùng trong bài là `A_in.csv`. Notebook này trình bày riêng thuật toán A* Euclid cho đề 2 năm 2025.

## Đề bài và dữ liệu đầu vào

Dòng đầu của `A_in.csv` là:

```text
10,5,5
```

Ý nghĩa:

- `n = 10`: lâu đài là ma trận `10 x 10`.
- Vị trí bắt đầu là `(5,5)`.
- Tọa độ dùng kiểu `0-based`, tức dòng/cột bắt đầu từ `0`.

Mục tiêu là tìm một đường đi từ `(5,5)` đến một ô bất kỳ nằm trên rìa ma trận.

## Định nghĩa khoảng cách Euclid và công thức h(x)

Khoảng cách Euclid là khoảng cách đường thẳng giữa hai điểm trong mặt phẳng.

Với hai điểm:

```text
A = (x1, y1)
B = (x2, y2)
```

công thức khoảng cách Euclid là:

```text
d(A, B) = sqrt((x1 - x2)^2 + (y1 - y2)^2)
```

Trong bài lâu đài, mục tiêu là một ô bất kỳ nằm trên rìa ma trận. Với một ô hiện tại:

```text
x = (row, col)
```

khoảng cách Euclid ngắn nhất từ ô đó đến bốn cạnh của ma trận được tính bằng:

```text
h(x) = min(
    sqrt(row^2),
    sqrt(col^2),
    sqrt((n - 1 - row)^2),
    sqrt((n - 1 - col)^2)
)
```

Vì các giá trị trong căn đều không âm sau khi bình phương, công thức này có giá trị tương đương:

```text
h(x) = min(row, col, n - 1 - row, n - 1 - col)
```

Tuy nhiên trong đề 2, code vẫn viết theo dạng Euclid bằng `math.sqrt(...)` để thể hiện đúng yêu cầu dùng khoảng cách Euclid.

Ví dụ với `n = 10`, ô bắt đầu `(5,5)`:

```text
Đến rìa trên:  sqrt(5^2) = 5
Đến rìa trái:  sqrt(5^2) = 5
Đến rìa dưới:  sqrt((9 - 5)^2) = 4
Đến rìa phải:  sqrt((9 - 5)^2) = 4

h(5,5) = 4
```

Nếu xét ô `(7,9)`:

```text
Đến rìa phải: sqrt((9 - 9)^2) = 0
h(7,9) = 0
```

Vì `h(7,9) = 0`, ô `(7,9)` nằm trên rìa và là một cửa ra hợp lệ.

## Biểu diễn bài toán dưới dạng đồ thị

Bài toán lâu đài có thể xem như một bài toán tìm đường trên đồ thị:

- Mỗi ô trong ma trận là một đỉnh.
- Chỉ những ô có giá trị `1` mới là đỉnh hợp lệ vì đó là ô có đường hầm.
- Hai đỉnh có cạnh nối với nhau nếu hai ô tương ứng kề cạnh theo một trong bốn hướng: trên, phải, dưới, trái.
- Ô bắt đầu là `(5,5)`.
- Tập đích là các ô đi được nằm trên rìa ma trận.

Chương trình không tạo sẵn toàn bộ danh sách cạnh. Khi đang xét một ô, hàm `get_neighbors()` sinh ra các ô kề hợp lệ, nên đồ thị được suy ra trực tiếp từ ma trận đầu vào.

## Code chương trình A* Euclid

Dưới đây là chương trình hiện có trong file `cau1_astar.py`. Notebook chỉ sao chép nội dung để trình bày, không chỉnh sửa file code gốc.

In [ ]:
from pathlib import Path
import csv
import heapq
import math

import matplotlib.pyplot as plt


def read_input(file_path):
    with open(file_path, newline="", encoding="utf-8-sig") as f:
        reader = csv.reader(f)
        first_line = next(reader)
        n = int(first_line[0])
        start = (int(first_line[1]), int(first_line[2]))
        maze = [[int(value) for value in row] for row in reader]

    return n, start, maze


def heuristic_to_border(position, n):
    row, col = position
    distances = [
        math.sqrt(row**2),
        math.sqrt(col**2),
        math.sqrt((n - 1 - row) ** 2),
        math.sqrt((n - 1 - col) ** 2),
    ]
    return min(distances)


def is_inside(position, n):
    row, col = position
    return 0 <= row < n and 0 <= col < n


def is_border(position, n):
    row, col = position
    return row == 0 or row == n - 1 or col == 0 or col == n - 1


def get_neighbors(position, n, maze):
    row, col = position
    directions = [(-1, 0), (0, 1), (1, 0), (0, -1)]
    neighbors = []

    for d_row, d_col in directions:
        next_pos = (row + d_row, col + d_col)

        if is_inside(next_pos, n) and maze[next_pos[0]][next_pos[1]] == 1:
            neighbors.append(next_pos)

    return neighbors


def reconstruct_path(parent, goal):
    path = []
    current = goal

    while current is not None:
        path.append(current)
        current = parent[current]

    path.reverse()
    return path


def astar_escape(n, start, maze):
    if not is_inside(start, n) or maze[start[0]][start[1]] != 1:
        return None

    open_set = []
    start_g = 0
    start_h = heuristic_to_border(start, n)
    heapq.heappush(open_set, (start_g + start_h, start_h, start_g, start))

    parent = {start: None}
    g_score = {start: 0}
    visited = set()

    while open_set:
        _, _, current_g, current = heapq.heappop(open_set)

        if current in visited:
            continue

        visited.add(current)

        if is_border(current, n):
            return reconstruct_path(parent, current)

        for neighbor in get_neighbors(current, n, maze):
            tentative_g = current_g + 1

            if neighbor not in g_score or tentative_g < g_score[neighbor]:
                g_score[neighbor] = tentative_g
                parent[neighbor] = current
                h = heuristic_to_border(neighbor, n)
                f = tentative_g + h
                heapq.heappush(open_set, (f, h, tentative_g, neighbor))

    return None


def write_output(file_path, path):
    with open(file_path, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.writer(f)

        if path is None:
            writer.writerow([-1])
            return

        writer.writerow([len(path)])
        writer.writerows(path)


def save_path_chart(maze, path, output_file):
    n = len(maze)

    plt.figure(figsize=(6, 6))
    plt.imshow(maze, cmap="gray_r")
    plt.xticks(range(n))
    plt.yticks(range(n))
    plt.grid(color="lightgray", linewidth=0.5)

    if path is not None:
        rows = [position[0] for position in path]
        cols = [position[1] for position in path]
        plt.plot(cols, rows, color="red", linewidth=2.5, marker="o", markersize=5, label="Duong di A*")
        plt.scatter(cols[0], rows[0], c="lime", s=140, edgecolors="black", label="Bat dau")
        plt.scatter(cols[-1], rows[-1], c="yellow", s=140, edgecolors="black", label="Cua ra")

    plt.title("Duong thoat tim duoc bang A*")
    plt.xlabel("Cot")
    plt.ylabel("Dong")
    plt.legend(loc="upper right", fontsize=8)
    plt.tight_layout()
    plt.savefig(output_file, dpi=160)
    plt.close()


def main():
    current_dir = Path(__file__).resolve().parent
    input_file = current_dir.parent / "A_in.csv"
    output_file = current_dir / "A_out.csv"
    path_image = current_dir / "cau1_astar_path.png"

    n, start, maze = read_input(input_file)
    path = astar_escape(n, start, maze)
    write_output(output_file, path)
    save_path_chart(maze, path, path_image)

    if path is None:
        print("Khong tim thay duong thoat. A_out.csv ghi -1.")
    else:
        print(f"Tim thay duong thoat bang A*: {len(path)} o.")
        print(f"Da ghi ket qua vao: {output_file}")
        print(f"Da luu bieu do duong di: {path_image}")


if __name__ == "__main__":
    main()

## Giải thích tác dụng của từng hàm

- `read_input(file_path)`: đọc file CSV đầu vào, lấy kích thước ma trận `n`, vị trí bắt đầu `start` và ma trận lâu đài `maze`.
- `heuristic_to_border(position, n)`: tính `h(x)`, tức khoảng cách Euclid ngắn nhất từ ô hiện tại đến rìa gần nhất. A* dùng `h` trong `f = g + h`, còn Greedy dùng trực tiếp `h` để ưu tiên ô tiếp theo.
- `is_inside(position, n)`: kiểm tra một tọa độ có nằm trong phạm vi ma trận `n x n` hay không.
- `is_border(position, n)`: kiểm tra một ô có nằm ở rìa lâu đài hay không. Nếu có, ô đó là cửa ra hợp lệ.
- `get_neighbors(position, n, maze)`: sinh các ô kề cạnh hợp lệ theo bốn hướng trên, phải, dưới, trái. Chỉ những ô nằm trong ma trận và có giá trị `1` mới được đi tiếp.
- `reconstruct_path(parent, goal)`: truy vết đường đi từ ô cửa ra về ô bắt đầu dựa trên dictionary `parent`, sau đó đảo ngược lại để có đường đi đúng chiều.
- `astar_escape(n, start, maze)`: thực hiện A*. Hàm dùng hàng đợi ưu tiên `open_set`, chọn ô có `f = g + h` nhỏ nhất, cập nhật chi phí tốt hơn cho các ô kề và trả về đường thoát nếu gặp rìa.
- `write_output(file_path, path)`: ghi kết quả ra file CSV. Nếu không tìm thấy đường thì ghi `-1`, nếu tìm thấy thì ghi số ô của đường đi và danh sách tọa độ.
- `save_path_chart(maze, path, output_file)`: vẽ ma trận lâu đài và đường đi tìm được, sau đó lưu thành ảnh PNG.
- `main()`: hàm điều phối chính, đọc input, chạy thuật toán, ghi output, lưu biểu đồ và in thông báo kết quả.

## Giải thích thuật toán A*

A* là thuật toán tìm kiếm có thông tin. Trong đề 2, A* dùng khoảng cách Euclid để ước lượng ô hiện tại còn cách rìa gần nhất bao xa.

Công thức đánh giá:

```text
f(x) = g(x) + h(x)
```

Trong đó `g(x)` là số bước thật từ `(5,5)` đến ô `x`, còn `h(x)` là khoảng cách Euclid ước lượng từ ô `x` đến rìa gần nhất.

Quy trình thuật toán:

1. Kiểm tra ô bắt đầu có hợp lệ và đi được không.
2. Khởi tạo `open_set` là hàng đợi ưu tiên, đưa ô bắt đầu vào với `g = 0`, `h = heuristic_to_border(start, n)`, `f = g + h`.
3. Khởi tạo `parent`, `g_score` và `visited`.
4. Trong vòng lặp, lấy ô có `f` nhỏ nhất ra khỏi `open_set`.
5. Nếu ô đó nằm ở rìa, gọi `reconstruct_path()` và dừng.
6. Nếu chưa tới rìa, sinh các ô kề hợp lệ bằng `get_neighbors()`.
7. Với mỗi ô kề, tính `tentative_g = current_g + 1`. Nếu chi phí mới tốt hơn, cập nhật `g_score`, cập nhật `parent`, tính `h`, tính `f`, rồi đưa vào `open_set`.
8. Nếu `open_set` rỗng mà chưa gặp rìa, kết luận không có đường thoát.

## Phân tích chi tiết luồng xử lý

Trước tiên, chương trình kiểm tra ô bắt đầu `(5,5)` có nằm trong ma trận và có đi được không. Nếu ô bắt đầu không hợp lệ thì trả về `None`.

Sau đó thuật toán khởi tạo cấu trúc dữ liệu chính: `open_set` ưu tiên theo `f = g + h`; `g_score` lưu chi phí tốt nhất; `parent` lưu đường đi; `visited` tránh xử lý lại ô đã xét.

Ở mỗi vòng lặp, thuật toán lấy một ô ra xử lý, kiểm tra ô đó có nằm trên rìa không. Nếu đã ở rìa, chương trình gọi `reconstruct_path()` để dựng lại đường đi. Nếu chưa tới rìa, chương trình gọi `get_neighbors()` để sinh các ô kề hợp lệ rồi cập nhật cấu trúc chờ xét.

## Bảng bước chạy chi tiết

| Bước | Ô lấy ra | g | h | f | Cập nhật | Open set sau bước | Ghi chú |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | `(5,5)` | 0 | 4 | 4 | `(4,5) g=1 h=4 f=5`, `(5,6) g=1 h=3 f=4` | `[(5,6) f=4, (4,5) f=5]` | Tiếp tục |
| 2 | `(5,6)` | 1 | 3 | 4 | `(5,7) g=2 h=2 f=4` | `[(5,7) f=4, (4,5) f=5]` | Tiếp tục |
| 3 | `(5,7)` | 2 | 2 | 4 | `(5,8) g=3 h=1 f=4` | `[(5,8) f=4, (4,5) f=5]` | Tiếp tục |
| 4 | `(5,8)` | 3 | 1 | 4 | `(4,8) g=4 h=1 f=5`, `(6,8) g=4 h=1 f=5` | `[(4,8) f=5, (6,8) f=5, (4,5) f=5]` | Tiếp tục |
| 5 | `(4,8)` | 4 | 1 | 5 | `(3,8) g=5 h=1 f=6` | `[(6,8) f=5, (4,5) f=5, (3,8) f=6]` | Tiếp tục |
| 6 | `(6,8)` | 4 | 1 | 5 | `(7,8) g=5 h=1 f=6` | `[(4,5) f=5, (3,8) f=6, (7,8) f=6]` | Tiếp tục |
| 7 | `(4,5)` | 1 | 4 | 5 | `(3,5) g=2 h=3 f=5` | `[(3,5) f=5, (3,8) f=6, (7,8) f=6]` | Tiếp tục |
| 8 | `(3,5)` | 2 | 3 | 5 | `(3,4) g=3 h=3 f=6` | `[(3,8) f=6, (7,8) f=6, (3,4) f=6]` | Tiếp tục |
| 9 | `(3,8)` | 5 | 1 | 6 | `(2,8) g=6 h=1 f=7` | `[(7,8) f=6, (3,4) f=6, (2,8) f=7]` | Tiếp tục |
| 10 | `(7,8)` | 5 | 1 | 6 | `(7,9) g=6 h=0 f=6` | `[(7,9) f=6, (3,4) f=6, (2,8) f=7]` | Tiếp tục |
| 11 | `(7,9)` | 6 | 0 | 6 |  | `[(3,4) f=6, (2,8) f=7]` | Gặp cửa ra |

## Biểu đồ đường đi

![Đường thoát tìm được](cau1_astar_path.png)

Trong hình minh họa:

- Ô có giá trị `1` là ô đi được.
- Ô có giá trị `0` là tường hoặc không có đường hầm.
- Đường màu đỏ nối các tọa độ trong danh sách đường đi.
- Điểm đầu là `(5,5)`, tức phòng trung tâm.
- Điểm cuối là ô nằm trên rìa ma trận, tức cửa ra.

Đường đi trong file output là:

`(5,5)` -> `(5,6)` -> `(5,7)` -> `(5,8)` -> `(6,8)` -> `(7,8)` -> `(7,9)`

Dòng đầu của file output là `7`, nghĩa là đường đi có `7` ô. Nếu tính số bước di chuyển thì số bước bằng `7 - 1`.

## Kết quả thực thi

Lệnh chạy chương trình:

```powershell
python 2025/De2/BFS_DFS_Astar_Cau1/01_AStar/cau1_astar.py
```

Kết quả trong `A_out.csv`:

```text
7
5,5
5,6
5,7
5,8
6,8
7,8
7,9
```

Kết luận: A* Euclid tìm được đường thoát hợp lệ từ `(5,5)` đến một ô ở rìa ma trận.